In [24]:
import pandas as pd
import numpy as np
import logging
from config import TABLE_LIST
import pymysql
from datetime import datetime
from dotenv import load_dotenv
import os
import json


import requests
import pyarrow.parquet as pq
import pyarrow as pa
from sqlalchemy import create_engine, text
from sqlalchemy.pool import QueuePool

In [2]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

### At first we use the PyMysql engine to extract data and load the data directly to file

First I configure mysql / mariadb to accept connections from any IP (could be dangerous) in /etc/mysq/mariadb.conf.d/50-server.cnf 


bind-address = 0.0.0.0

Then I create a read-only user and grant SELECT privileges only to that user for the database whose data is to be extracted:

```CREATE USER 'etl_user'@'%' IDENTIFIED BY 'mypass980898'; FLUSH PRIVILEGES;```

```GRANT SELECT ON database_name.* TO 'etl_user'@'%';```

In [3]:
load_dotenv(".env")

host = os.getenv("HOST")
port = os.getenv("PORT")
user = os.getenv("DB_USER")
db = os.getenv("DB_NAME")
password = os.getenv("PASSWORD")
api_key = os.getenv("API_KEY")
api_secret = os.getenv("API_SECRET")


In [4]:
def test_get_customer():
    BASE_URL = "https://tst.neviraminerals.com/"

    headers = {
        "Authorization": f"token {api_key}:{api_secret}",
        "Content": "application/json"
    }

    params = {
        "page": 1,
        "page_length": 40
    }

    URL = f"{BASE_URL}api/method/neviraflow.api.get_customer_list"
    response = requests.get(URL,params = params, headers=headers)
    if response.status_code == 200:
        print(response)
    
        data = response.json()
        return data
    else:
        print("Failed to connect to URL")

In [5]:
res = test_get_customer()

<Response [200]>


In [6]:
list_of_dicts = res["message"]["data"]
df_res = pd.DataFrame(list_of_dicts)

In [7]:
df_res_ = pd.json_normalize(list_of_dicts)

In [8]:
def get_one_dict(list_dict: list):
    keys = list_of_dicts[0].keys()
    out_dict = {key:[] for key in keys}
    for dict_ in list_of_dicts:
        for key, value in dict_.items():
            out_dict[key].append(value)
    return out_dict

In [9]:
logging.info("Connecting to remote database ....")
connection_config_remote = {
    "host": host,
    "user": user,
    "database":db,
    "port": int(port),
    "password":password
}

## pymysql does not privide connection pooling so we have to open the connection and close it
## every time we need to make a database connection
with pymysql.connect(**connection_config_remote) as connection:
    query = """ SELECT 
                    name, 
                    customer, 
                    customer_name, 
                    posting_date, 
                    due_date, 
                    base_grand_total FROM `tabSales Invoice` WHERE docstatus = 1 
                    AND YEAR(posting_date) = 2026 """
    logging.info("Extracting data ....")
    df_remote = pd.read_sql(query, connection)
    today_date = datetime.strftime(datetime.today(), "%Y-%m-%d %HH-%MM-%SS")
    logging.info("Loading data to file storage")
    df_remote.to_parquet(f"data/sales_invoice_{today_date}")

2026-09-11 15:58:13,961 - INFO - Connecting to remote database ....
2026-09-11 15:58:15,010 - INFO - Extracting data ....
/tmp/ipykernel_65213/3248204510.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_remote = pd.read_sql(query, connection)
2026-09-11 15:58:15,864 - INFO - Loading data to file storage


In [10]:
df_remote.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2112 entries, 0 to 2111
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   name              2112 non-null   object 
 1   customer          2112 non-null   object 
 2   customer_name     2112 non-null   object 
 3   posting_date      2112 non-null   object 
 4   due_date          2112 non-null   object 
 5   base_grand_total  2112 non-null   float64
dtypes: float64(1), object(5)
memory usage: 99.1+ KB


### Now we move towards extracting the data using SQLAlchemy

SQLAlchemy helps us manage database connections, create an abstraction layer, handles lazy loading and relationship management.

In [26]:
## database connection manager / factory to handle connection to a database i.e mysql, oracle, postgresql


## create engine with connection pooling
mysql_engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}/{db}",
                            poolclass = QueuePool,
                            pool_size=10, 
                            pool_pre_ping=True, 
                            echo=False
                        )


In [ ]:

query = text("""SELECT 
                    name, 
                    supplier, 
                    supplier_name, 
                    posting_date, 
                    due_date, 
                    base_grand_total FROM `tabPurchase Invoice` WHERE docstatus = 1 
                    AND YEAR(posting_date) = 2026 """)

with mysql_engine.connect() as connection:
    df_a = pd.read_sql(query, connection)
    extraction_date = datetime.strftime(datetime.today(), "%Y-%m-%d")
    df_a.to_parquet(f"data/purchase_invoice_{extraction_date}")

In [13]:
datetime.strftime(datetime.now(),"%Y_%m_%d_%H_%M_%S")

'2026_09_11_15_58_25'

Chunking reduces the amount of data held in memory at one given time,
but it does not reduce the total amount of data extracted.

Improvements will need to made for the following scenarios:

- When we have a new table and it has never been extracted yet
- Doing incremental loading while making sure that the new tables are not affected by the date watermark
- Is my data pipeline idempotent ? 


In [ ]:
erp_table_names = TABLE_LIST
with mysql_engine.connect() as connection:
    for erp_table in erp_table_names:
        select_query = text(f""" SELECT * FROM `tab{erp_table}` """)

        table_name_clean = erp_table.replace(" ","")

        table_folder = f"data/{table_name_clean}"

        ## we want to have a sub-directory for every table
        os.makedirs(table_folder, exist_ok=True)
        
        logging.info(f" Extracting data from {erp_table} ..... ")
        extraction_date_time = datetime.strftime(datetime.now(),"%Y_%m_%d_%H_%M_%S")

        parquet_path = f"{table_folder}/{extraction_date_time}.parquet"

        ## read the data in tables in batches instead of everything at once
        chunks = pd.read_sql(select_query,
                            connection, 
                            chunksize=10000 ## meaning we will extract only 10,000 rows at a time
                        )

        parquet_writer = None

        n_rows = 0

        try:
            for chunk in chunks:
                ## Use the schema of the first chunk to create the parquet writer

                if parquet_writer is None:
                    parquet_schema = pa.Table.from_pandas(chunk).schema
                    parquet_writer = pq.ParquetWriter(parquet_path, parquet_schema)

                parquet_table = pa.Table.from_pandas(chunk)
                parquet_writer.write(parquet_table)
                n_rows += len(chunk)

                logger.info(f"Written a total of {n_rows} rows for table {table_name_clean}")

            if n_rows == 0:
                logger.warning("No rows returned by the query!")
                
        except Exception as e:
            logger.error(f"An error occured during data extraction ....{e}")
            raise ## Stop the execution right here if any failure is encountered

        ## Close the parquet writer
        finally:
            if parquet_writer is not None:
                parquet_writer.close()

    execution_end_time = datetime.now()

    ## Save the execution end time to a file, we will use it for incremental loading the next run
    with open("execution_reads.txt","a") as execution_file:
        execution_file.write(f"{execution_end_time} \n")

2026-09-11 16:25:54,327 - INFO -  Extracting data from Customer ..... 
2026-09-11 16:25:55,012 - INFO - Written a total of 339 rows for table Customer
2026-09-11 16:25:55,016 - INFO -  Extracting data from Sales Team ..... 
2026-09-11 16:25:55,930 - INFO - Written a total of 5081 rows for table SalesTeam
2026-09-11 16:25:55,932 - INFO -  Extracting data from Supplier ..... 
2026-09-11 16:25:56,309 - INFO - Written a total of 124 rows for table Supplier
2026-09-11 16:25:56,313 - INFO -  Extracting data from Contact ..... 
2026-09-11 16:25:56,677 - INFO - Written a total of 216 rows for table Contact
2026-09-11 16:25:56,680 - INFO -  Extracting data from Address ..... 
2026-09-11 16:25:56,941 - INFO - Written a total of 254 rows for table Address
2026-09-11 16:25:56,944 - INFO -  Extracting data from Employee ..... 
2026-09-11 16:25:57,245 - INFO - Written a total of 0 rows for table Employee
2026-09-11 16:25:57,247 - WARNING - No rows returned by the query!
2026-09-11 16:25:57,251 - INF

In [17]:
len(TABLE_LIST)

37

### Incremental loading

Refers to moving or copying only data that is new or data that has changed from the data source 
since the last run, rather than extracting the entire data source.

Techniques for incremental loading include:
    
- Timestamps: tracks modification times
- Change Data Capture (CDC): CDC systems directly monitor changes (inserts, updates, deletes) in the source database by reading transasction logs

We will use the timestamp approach. ERPNext tables have a modified field, from which we will extract the maximum modified date
and use it to watermark the tables
The next extract qeries will be `SELECT * FROM table_name WHERE modified > water_mark_date`

In [30]:
METADATA_FOLDER = "metadata"

def get_watermark(table_name):
    metadata_path = f"{METADATA_FOLDER}/{table_name}.json"

    if not os.path.exists(metadata_path):  ## If it does not exist it means the extraction has not happended for that table yet hence do a full load
        return None

    else:
        with open(metadata_path, "r") as metadata_file:
            metadata = json.load(metadata_file)   ## read the json file
        return metadata["watermark"] ## get the max modified date

def save_watermark(table_name, watermark):
    os.makedirs(METADATA_FOLDER, exist_ok=True)
    metadata_path = f"{METADATA_FOLDER}/{table_name}.json"
    metadata = {
        "table": table_name,
        "watermark": str(watermark)
    }

    with open(metadata_path,"w") as metadata_file:
        json.dump(metadata,metadata_file, indent=4)

In [31]:
test_table_name = "TestOrders"
save_watermark(test_table_name,datetime.strftime(datetime.now(),"%Y-%m-%d %HH:%MM:%SS"))

In [28]:
erp_table_names = TABLE_LIST

with mysql_engine.connect() as connection:
    for erp_table in erp_table_names:

        table_name_clean = erp_table.replace(" ","")

        watermark = get_watermark(table_name_clean)  ## get the max modified date in the specific table

        if watermark is None:
            select_query = text(f""" SELECT * FROM `tab{erp_table}` """)

        else:
            select_query = text(f""" SELECT * FROM `{erp_table}` WHERE modified > :watermark """ )

        

        table_folder = f"data/{table_name_clean}"

        ## we want to have a sub-directory to store parquet files for every table
        os.makedirs(table_folder, exist_ok=True)

        ## keep track of the extraction time
        extraction_date_time = datetime.strftime(datetime.now(),"%Y_%m_%d_%H_%M_%S")

        parquet_path = f"{table_folder}/{extraction_date_time}.parquet"

        ## read the data in tables in batches instead of everything at once, and also take care of the last extract date (watermark) if the watermark exists
        if watermark is None:
            logging.info(f"First extraction for {erp_table}")
            chunks = pd.read_sql(select_query,
                                connection, 
                                chunksize=10000) ## meaning we will extract only 10,000 rows at a time

        else:
            logging.info(f"Incremental extraction for {erp_table}, watermark: {watermark}")
            chunks = pd.read_sql(select_query,
                                connection,
                                params={"watermark":watermark},
                                chunksize = 10000)

        parquet_writer = None

        n_rows = 0
        max_modified = None
        
        try:
            for chunk in chunks:

                ## get the maximum modified date from each chunk
                chunk_max_modified = chunk["modified"].max()

                ## we will keep track of the maximum modified value
                if max_modified is None or chunk_max_modified > max_modified:
                    max_modified = chunk_max_modified

                ## Use the schema of the first chunk to create the parquet writer
                if parquet_writer is None:
                    parquet_schema = pa.Table.from_pandas(chunk).schema
                    parquet_writer = pq.ParquetWriter(parquet_path, parquet_schema)

                parquet_table = pa.Table.from_pandas(chunk)
                parquet_writer.write(parquet_table)
                n_rows += len(chunk)

                logger.info(f"Written a total of {n_rows} rows for table {table_name_clean}")


            if max_modified is not None:
                save_watermark(table_name_clean, max_modified)

            if n_rows == 0:
                logger.warning("No rows returned by the query!")
                
        except Exception as e:
            logger.error(f"An error occured during data extraction ....{e}")
            raise ## Stop the execution right here if any failure is encountered

        ## Close the parquet writer
        finally:
            if parquet_writer is not None:
                parquet_writer.close()

    execution_end_time = datetime.now()

    ## Save the execution end time to a file
    with open("execution_reads.txt","a") as execution_file:
        execution_file.write(f"{execution_end_time} \n")

2026-09-11 19:54:51,586 - INFO - First extraction for Customer
2026-09-11 19:54:52,600 - INFO - Written a total of 339 rows for table Customer
2026-09-11 19:54:52,602 - INFO - First extraction for Sales Team
2026-09-11 19:55:03,157 - INFO - Written a total of 5081 rows for table SalesTeam
2026-09-11 19:55:03,158 - INFO - First extraction for Supplier
2026-09-11 19:55:03,758 - INFO - Written a total of 124 rows for table Supplier
2026-09-11 19:55:03,760 - INFO - First extraction for Contact
2026-09-11 19:55:04,416 - INFO - Written a total of 216 rows for table Contact
2026-09-11 19:55:04,418 - INFO - First extraction for Address
2026-09-11 19:55:05,361 - INFO - Written a total of 254 rows for table Address
2026-09-11 19:55:05,362 - INFO - First extraction for Employee
2026-09-11 19:55:05,586 - INFO - Written a total of 0 rows for table Employee
2026-09-11 19:55:05,587 - WARNING - No rows returned by the query!
2026-09-11 19:55:05,588 - INFO - First extraction for Sales Order
2026-09-11 

KeyboardInterrupt: 

#### What happens when there is timeout, what are the retry mechanisms for this pipeline ? 


In [22]:
### Parametized query to get data using parameters
def get_invoice_by_customer(customer_id):
    query = text("""
                    SELECT si.name,
                           si.customer_name,
                           si.posting_date, 
                           si.due_date,
                           si.payment_terms_template,
                           si.base_grand_total,
                           si.outstanding_amount,
                           si.status
                    FROM `tabSales Invoice` AS si 
                    WHERE si.customer = :customer_id
                    AND si.docstatus = 1
                    ORDER BY si.posting_date DESC
                """)


    ## Context manager 
    with mysql_engine.connect() as connection:
        
        result = connection.execute(query, {"customer_id":customer_id}) ## Applying a filter to the query
        df = pd.DataFrame(result.fetchall(), columns = result.keys())
    return df
        
    

In [ ]:
def individual_sales_report():
    query = text(""" 
                    SELECT 
                        si.name AS invoice_number,
                        si.posting_date,
                        si.customer,
                        si.customer_name,
                        st.sales_person,
                        si.due_date,
                        si.currency,
                        si.payment_terms_template,
                        sii.item_code,
                        sii.item_name, 
                        sii.rate,
                        sii.qty,
                        ROUND(si.base_grand_total,2) AS base_grand_total,
                        ROUND(si.outstanding_amount,2) AS outstanding_amount,
                        si.status
                    FROM `tabSales Invoice Item` AS sii 
                        INNER JOIN `tabSales Invoice` AS si ON sii.parent = si.name
                        LEFT JOIN `tabSales Team` AS st ON si.customer = st.parent
                    WHERE si.docstatus = 1 
                        AND si.is_opening = 0
                        AND si.is_return = 0
                        AND si.status NOT IN ('Credit Note Issued')
                    ORDER BY si.posting_date DESC
                    """)
    with mysql_engine.connect() as connection:
        df = pd.read_sql(query, connection)
        
    if not df.empty:
        ## transformation step
        df['amount_paid'] = df['base_grand_total'] -  df['outstanding_amount']

    return df

In [26]:
item_wise_sales = individual_sales_report()

### Extracting large databases / tables with batch processing and exception handling

In [28]:
import logging
from sqlalchemy.exc import SQLAlchemyError, OperationalError

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def fetch_stock_ledger_entry():
    query = text(""" SELECT * FROM `tabStock Ledger Entry` WHERE is_cancelled = 0 ORDER BY creation DESC""")

    try:
        with engine.connect() as connection:
            results = connection.execution_options(stream_results=True).execute(query)
    
            ## Process the data in chunks
            chunk_size = 10000
    
            chunks = []
    
            while True:
                chunk = results.fetchmany(chunk_size)
    
                if not chunk:
                    break
                chunks.append(pd.DataFrame(chunk, columns = results.keys()))
            
            sle_df =  pd.concat(chunks, ignore_index=True)

            ## get the query execution time
            run_time = datetime.strftime(datetime.today(), "%Y-%m-%d")
            
            ## Save dataframe to parquet
            sle_df.to_parquet(f"data/stock_ledger_entry_{run_time}.parquet")

            return sle_df
            
            
    except OperationalError as e:
        logger.error(f"Database operation error: {e}")
        return None
        
    except SQLAlchemyError as e:
        logger.error(f"SQLAlchemy error: {e}")
        return None

    except Exception as e:
        logger.error(f"Unexected error occured: {e}")
        return None
        

In [29]:
sle_df = fetch_stock_ledger_entry()

In [30]:
sle_df.shape

(174717, 45)